# Transfection analysis

Edit the **Config** cell, then run setup and each step: segment, traces, AUC, fit.

The assay comes from Studio Metadata: samples, channels, interval, max onset time, and Skip segmentation. This notebook does not write those fields. Segment uses Otsu (variation radius 2, Gaussian sigma 1) when Skip segmentation is off, which is the Studio default. When it is on, traces use the full crop and a 10th-percentile background.

Sample names and plots are written in `results.ipynb` from the same `assay.json`.


## Config


In [ ]:
from pathlib import Path

# Folder with roi/ from crop and assay.json from Studio.
WORKSPACE = Path(r"Z:\projects\LNPbinder\Experiments\20260731\Auswertung")


## Setup


In [ ]:
from lisca.core.assay_positions import load_assay_positions
from transfection.core import SampleAnalysis, load_assay_for_workspace

workspace = WORKSPACE.expanduser().resolve()
if not workspace.is_dir():
    raise FileNotFoundError(f"Workspace not found: {workspace}")

positions = load_assay_positions(workspace)
config = load_assay_for_workspace(workspace)
if config.assay_type != "transfection":
    raise ValueError(
        f"assay.json type is {config.assay_type!r}. "
        "These notebooks run the transfection pipeline. Analyze this assay in Studio."
    )
if not config.mapping:
    raise ValueError("assay.json has no samples. Name each sample on Studio Metadata.")

mapping = {}
for name, entry in config.mapping.items():
    file_ids = positions.samples.get(name)
    if not file_ids:
        raise ValueError(f"Sample {name!r} has no positions in assay.json.")
    mapping[name] = SampleAnalysis(
        name=entry.name,
        positions=file_ids,
        signal_channels=list(entry.signal_channels),
        segmentation_channel=entry.segmentation_channel,
    )

print(f"Workspace: {workspace}")
print(f"Interval: {config.interval_minutes} minutes")
print(f"Max onset time t0: {config.max_onset_minutes} minutes")
print(f"Samples: {', '.join(mapping)}")
if config.skip_segment:
    print(
        "Skip segmentation is on. Traces use the full crop and a "
        "10th-percentile background."
    )
else:
    print(
        "Skip segmentation is off. Segmenting with Otsu, variation radius 2, "
        "Gaussian sigma 1."
    )


## Segment

Otsu masks, matching Studio. Skipped when Metadata has Skip segmentation on.


In [ ]:
from transfection.services import segment

if config.skip_segment:
    print("Segmentation skipped.")
else:
    result = segment.run_segment(
        workspace=workspace,
        mapping=mapping,
        variation_radius=2,
        gaussian_sigma=1.0,
    )
    for sample, output_dir, mask_count in result.written_outputs:
        print(segment.format_written_masks_message(sample, output_dir, mask_count))
    if result.skipped_positions:
        print(segment.format_skipped_positions_message(result.skipped_positions))


## Traces


In [ ]:
from transfection.services import traces

result = traces.run_traces(
    workspace=workspace,
    mapping=mapping,
    skip_segment=config.skip_segment,
    on_csv_written=lambda pos, path, rows: print(
        traces.format_written_trace_csv_message(pos, path, rows)
    ),
)
if result.skipped_positions:
    print(traces.format_skipped_positions_message(result.skipped_positions))


## AUC


In [ ]:
from transfection.services import auc

print(
    auc.format_written_auc_csv_message(
        auc.run_auc(workspace=workspace, interval=config.interval_minutes)
    )
)


## Fit


In [ ]:
from transfection.services import fit

print(
    fit.format_written_fit_csv_message(
        fit.run_fit(
            workspace=workspace,
            interval=config.interval_minutes,
            max_onset_minutes=config.max_onset_minutes,
        )
    )
)

print("Analysis finished. Continue in results.ipynb for tables and plots.")
